# Mapping of the leader-follower positions

**Notebook contents:**
* Extraction of needed data (model positions) for analysis
* Visualizations of radial spatial mapping 

In [ ]:
import numpy as np
import pandas as pd
import utils
import matplotlib.pyplot as plt

import os
from main import DorsognaTE
import joblib
from utils import set_plot_style

# Preliminaries

In [ ]:
phenotype_dict = {
    'singlemill':[0.5,0.1],
    'doublemill':[0.9,0.5],
    'doublering':[0.5,0.5],
    'collswarm':[0.5,0.9],
    'escapesymm':[2,0.9],
    'escapeunsymm':[2,2],
    'escapecoll':[2,3]
}

fixed_labels = utils.proper_phenotype_names

dorsogna_TE_exports_dir = 'csvs_actual_te_values'
dorsogna_positions_dir = 'simulations_dorsogna_info'
graphs_dir = 'graphs'


# Compile/extract data

## Compiling data from csvs

In [ ]:
TEdf_dict_linvel = dict()
for key, value in phenotype_dict.items():
    C, l = value
    TE_df = pd.read_csv(f'{dorsogna_TE_exports_dir}/{key}/TElog_{key}_{C}_{l}_linvel_k15.csv', index_col = 0)
    TEdf_dict_linvel[key] = TE_df

TEdf_dict_angvel = dict()
for key, value in phenotype_dict.items():
    C, l = value
    TE_df = pd.read_csv(f'{dorsogna_TE_exports_dir}/{key}/TElog_{key}_{C}_{l}_angvel_k15.csv', index_col = 0)
    TEdf_dict_angvel[key] = TE_df

## Save model positions

In [ ]:
for phenotype_name, values in phenotype_dict.items():
    C, l = values
    os.makedirs(dorsogna_positions_dir, exist_ok=True)
    insights_path = os.path.join(dorsogna_positions_dir, phenotype_name)
    model = DorsognaTE(outdir=insights_path)
    model.develop_model(
            C=C,
            l=l,
            phenotype_name=phenotype_name,
            particle_count=100,
            t_max=40,
            vel_scale=0.1,
            fps=20,
            dt=0.1,
            animate=False
        )

    joblib.dump(model.pos,f'{dorsogna_positions_dir}/{phenotype_name}/{phenotype_name}_positions.joblib')

# Run spatial mapping

In [ ]:
def compare_leader_follower_profiles(phenotype_name, pos_array, df, ver, timeslice=None, bins=100):
    if timeslice is None:
        timeslice = list(range(15, 400))
        
    # spatial pre-processing
    com = np.mean(pos_array[1:], axis=1)[:, np.newaxis, :]
    relative_pos = pos_array[1:] - com
    pos_sliced = relative_pos[timeslice]
    
    x = pos_sliced[:, :, 0].flatten()
    y = pos_sliced[:, :, 1].flatten()
    r = np.sqrt(x**2 + y**2)
    
    # process TE for leaders (source)
    df_leader = df.copy()
    df_leader.columns = [int(x.split('-')[0]) for x in df_leader.columns]
    TE_leader = df_leader.T.groupby(level=0).sum().T.to_numpy()[timeslice].flatten()

    # process TE for followers (destination)
    df_follower = df.copy()
    df_follower.columns = [int(x.split('-')[1]) for x in df_follower.columns]
    TE_follower = df_follower.T.groupby(level=0).sum().T.to_numpy()[timeslice].flatten()

    # radial binning
    r_bins = np.linspace(0, np.max(r), bins)
    r_indices = np.digitize(r, r_bins)
    
    radial_leader_te = []
    radial_follower_te = []
    
    for i in range(1, len(r_bins)):
        mask = (r_indices == i)
        if np.any(mask):
            radial_leader_te.append(np.mean(TE_leader[mask]))
            radial_follower_te.append(np.mean(TE_follower[mask]))
        else:
            radial_leader_te.append(0)
            radial_follower_te.append(0)

    ## visualization ##

    set_plot_style()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    # left plot: line graph
    ax1.plot(r_bins[:-1], radial_leader_te, label='Leader (Source)', color='#1f77b4', lw=2.5)
    ax1.plot(r_bins[:-1], radial_follower_te, label='Follower (Dest)', color='#ff7f0e', lw=2.5, linestyle='--')
    ax1.fill_between(r_bins[:-1], radial_leader_te, alpha=0.1, color='#1f77b4')
    ax1.fill_between(r_bins[:-1], radial_follower_te, alpha=0.1, color='#ff7f0e')
    
    ax1.set_title(f'Radial Information Profile: {fixed_labels[phenotype_name]}')
    ax1.set_xlabel('Distance from Swarm Center ($r$)')
    ax1.set_ylabel('Average Transfer Entropy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # right plot: right graph
    counts, xedges, yedges = np.histogram2d(x, y, bins=bins)
    sum_L, _, _ = np.histogram2d(x, y, bins=bins, weights=TE_leader)
    sum_F, _, _ = np.histogram2d(x, y, bins=bins, weights=TE_follower)

    with np.errstate(divide='ignore', invalid='ignore'):
        diff_heat = (sum_L - sum_F) / counts
        diff_heat = np.nan_to_num(diff_heat)

    # emphasize colors with percentile scalinge
    flat_diff = diff_heat.flatten()
    mask_has_data = counts.flatten() > 0
    valid_diffs = flat_diff[mask_has_data]
    
    if len(valid_diffs) > 0:
        vlimit_robust = np.percentile(np.abs(valid_diffs), 95) 
    else:
        vlimit_robust = 1 # Fallback
    
    vlimit_robust = max(vlimit_robust, 1e-6)
    pcm = ax2.imshow(diff_heat.T, origin='lower', 
                     extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
                     cmap='RdBu_r', interpolation='gaussian', 
                     vmin=-vlimit_robust, vmax=vlimit_robust)
    
    fig.colorbar(pcm, ax=ax2, label='Leader Surplus (Red) vs Follower Surplus (Blue)')
    ax2.set_title(f'Spatial Dominance')
    
    plt.tight_layout()

    # save    
    folder_name = f"{graphs_dir}/spatial_mapping"
    file_name = f"{phenotype_name}_{ver}_radialinfo_heatmap.png"

    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
        print(f"folder {folder_name} created")

    save_path = os.path.join(folder_name, file_name)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
for phenotype_name, value in phenotype_dict.items():
    C, l = value
    phenotype_name = phenotype_name
    
    model_pos = joblib.load(f'{dorsogna_positions_dir}/{phenotype_name}/{phenotype_name}_positions.joblib')
    model_pos.flags['WRITEABLE'] = True
    compare_leader_follower_profiles(phenotype_name = phenotype_name, 
                     pos_array = model_pos, 
                     df= TEdf_dict_linvel[phenotype_name], 
                     ver = 'linvel',
                     timeslice = list(range(15, 400)))
    compare_leader_follower_profiles(phenotype_name = phenotype_name, 
                    pos_array = model_pos, 
                    df= TEdf_dict_angvel[phenotype_name], 
                    ver = 'angvel',
                    timeslice = list(range(15, 400)))
